# Airbnb Location Tracking and Price Prediction Analysis

This notebook provides comprehensive analysis for Airbnb property investment decisions including:
- Price prediction using multiple regression models
- Market analysis and insights
- Revenue projections
- Actionable recommendations

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Import custom modules
from models.price_predictor import AirbnbPricePredictor
from utils.data_generator import generate_sample_data
from utils.insights_generator import generate_actionable_insights
from visualizations import AirbnbVisualizer

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All libraries imported successfully")

## 1. Configuration and Data Generation

Configure your location and generate sample Airbnb data for analysis.

In [ ]:
# Configuration - UPDATE THESE VALUES
LOCATION_CONFIG = {
    'latitude': 40.7128,    # New York City (example)
    'longitude': -74.0060,
    'radius_km': 5,         # 5 km radius
    'num_properties': 300   # Number of properties to analyze
}

# Your property configuration
YOUR_PROPERTY = {
    'property_type': 'Entire home/apt',
    'bedrooms': 2,
    'bathrooms': 2,
    'accommodates': 4,
    'latitude': LOCATION_CONFIG['latitude'],
    'longitude': LOCATION_CONFIG['longitude'],
    'rating': 4.5,
    'reviews_count': 50,
    'is_superhost': 1,
    'instant_bookable': 1
}

print("Configuration:")
print(f"  Location: ({LOCATION_CONFIG['latitude']}, {LOCATION_CONFIG['longitude']})")
print(f"  Radius: {LOCATION_CONFIG['radius_km']} km")
print(f"  Property Type: {YOUR_PROPERTY['property_type']}")
print(f"  Bedrooms: {YOUR_PROPERTY['bedrooms']}")

In [ ]:
# Generate sample data
print("Generating sample Airbnb data...")
df = generate_sample_data(
    latitude=LOCATION_CONFIG['latitude'],
    longitude=LOCATION_CONFIG['longitude'],
    radius_km=LOCATION_CONFIG['radius_km'],
    num_properties=LOCATION_CONFIG['num_properties']
)

print(f"\n✓ Generated {len(df)} properties")
print(f"\nDataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print("=" * 60)
print("MARKET STATISTICS")
print("=" * 60)

print(f"\nPrice Statistics:")
print(f"  Average Price: ${df['price'].mean():.2f}")
print(f"  Median Price: ${df['price'].median():.2f}")
print(f"  Min Price: ${df['price'].min():.2f}")
print(f"  Max Price: ${df['price'].max():.2f}")
print(f"  Std Dev: ${df['price'].std():.2f}")

print(f"\nProperty Types:")
print(df['property_type'].value_counts())

print(f"\nBedroom Distribution:")
print(df['bedrooms'].value_counts().sort_index())

print(f"\nAverage Ratings: {df['rating'].mean():.2f}")
print(f"Average Occupancy Rate: {df['occupancy_rate'].mean()*100:.1f}%")

In [ ]:
# Detailed statistics by property type
print("\n=" * 60)
print("STATISTICS BY PROPERTY TYPE")
print("=" * 60)

property_stats = df.groupby('property_type').agg({
    'price': ['mean', 'median', 'std', 'count'],
    'rating': 'mean',
    'occupancy_rate': 'mean'
}).round(2)

print(property_stats)

## 3. Data Visualizations

In [ ]:
# Initialize visualizer
visualizer = AirbnbVisualizer(df)

In [ ]:
# Price distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(df['price'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(df['price'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: ${df["price"].mean():.2f}')
axes[0].axvline(df['price'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: ${df["price"].median():.2f}')
axes[0].set_xlabel('Price ($)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Airbnb Prices', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot
axes[1].boxplot(df['price'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue', color='black'),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_ylabel('Price ($)', fontsize=12)
axes[1].set_title('Price Distribution (Box Plot)', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Price by property type
fig, ax = plt.subplots(figsize=(12, 6))

property_avg = df.groupby('property_type')['price'].mean().sort_values(ascending=False)
property_avg.plot(kind='bar', ax=ax, color='coral', edgecolor='black', alpha=0.8)

ax.set_xlabel('Property Type', fontsize=12)
ax.set_ylabel('Average Nightly Price ($)', fontsize=12)
ax.set_title('Average Price by Property Type', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
ax.grid(alpha=0.3, axis='y')

# Add value labels
for i, v in enumerate(property_avg):
    ax.text(i, v + 2, f'${v:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Monthly price trends
monthly_avg = df.groupby('month')['price'].mean()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(monthly_avg.index, monthly_avg.values, marker='o', linewidth=3, markersize=10, color='darkblue')
ax.fill_between(monthly_avg.index, monthly_avg.values, alpha=0.3, color='lightblue')

# Highlight peak and low
max_month = monthly_avg.idxmax()
min_month = monthly_avg.idxmin()
ax.scatter([max_month], [monthly_avg[max_month]], color='red', s=300, zorder=5, label=f'Peak: ${monthly_avg[max_month]:.2f}')
ax.scatter([min_month], [monthly_avg[min_month]], color='green', s=300, zorder=5, label=f'Low: ${monthly_avg[min_month]:.2f}')

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Average Price ($)', fontsize=12)
ax.set_title('Seasonal Price Trends', fontsize=14, fontweight='bold')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Geographic distribution
fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter(df['longitude'], df['latitude'], 
                     c=df['price'], s=80, alpha=0.6,
                     cmap='YlOrRd', edgecolors='black', linewidth=0.5)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Price ($)', rotation=270, labelpad=20, fontsize=12)

ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('Geographic Distribution of Properties', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = ['price', 'bedrooms', 'bathrooms', 'accommodates', 'rating', 'reviews_count', 'occupancy_rate']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, ax=ax, cbar_kws={'label': 'Correlation'})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Interactive scatter plot with Plotly
fig = px.scatter(df, x='bedrooms', y='price', color='property_type', 
                size='accommodates', hover_data=['rating', 'reviews_count'],
                title='Price vs Bedrooms by Property Type',
                labels={'bedrooms': 'Number of Bedrooms', 'price': 'Price ($)'},
                template='plotly_white', height=600)

fig.show()

## 4. Train Regression Models

Train multiple regression models to predict Airbnb prices.

In [ ]:
# Initialize and train the predictor
predictor = AirbnbPricePredictor()

print("Training multiple regression models...\n")
metrics = predictor.train(df)

print("\n" + "=" * 60)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 60)
print(f"\nBest Model: {metrics['best_model'].upper()}")
print(f"Training Set Size: {metrics['train_size']}")
print(f"Test Set Size: {metrics['test_size']}")

In [ ]:
# Display detailed metrics
metrics_df = pd.DataFrame(metrics['metrics']).T
metrics_df = metrics_df.sort_values('r2_score', ascending=False)

print("\nDetailed Model Metrics:")
print(metrics_df)

# Visualize model performance
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# R² Score comparison
metrics_df['r2_score'].plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_xlabel('R² Score', fontsize=12)
axes[0].set_title('Model R² Score Comparison', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3, axis='x')

# RMSE comparison
metrics_df['rmse'].plot(kind='barh', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_xlabel('RMSE ($)', fontsize=12)
axes[1].set_title('Model RMSE Comparison (Lower is Better)', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (if available)
if 'feature_importance' in metrics:
    importance_df = pd.DataFrame(metrics['feature_importance'])
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(importance_df['feature'], importance_df['importance'], color='green', edgecolor='black')
    ax.set_xlabel('Importance', fontsize=12)
    ax.set_title('Feature Importance', fontsize=14, fontweight='bold')
    ax.grid(alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
    
    print("\nTop 5 Most Important Features:")
    print(importance_df.head())

## 5. Price Predictions for Your Property

In [ ]:
# Predict monthly prices
print("=" * 60)
print("MONTHLY PRICE PREDICTIONS FOR YOUR PROPERTY")
print("=" * 60)

monthly_predictions = []
month_names = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']

for month in range(1, 13):
    features = YOUR_PROPERTY.copy()
    features['month'] = month
    
    predicted_price = predictor.predict(pd.DataFrame([features]))[0]
    monthly_predictions.append({
        'month': month,
        'month_name': month_names[month-1],
        'predicted_price': predicted_price
    })
    
    print(f"  {month_names[month-1]:12s}: ${predicted_price:.2f} / night")

monthly_pred_df = pd.DataFrame(monthly_predictions)
avg_price = monthly_pred_df['predicted_price'].mean()
print(f"\n  Average Annual Price: ${avg_price:.2f} / night")

In [ ]:
# Visualize monthly predictions
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(monthly_pred_df['month'], monthly_pred_df['predicted_price'], 
        marker='o', linewidth=3, markersize=10, color='darkgreen', label='Predicted Price')
ax.fill_between(monthly_pred_df['month'], monthly_pred_df['predicted_price'], 
                alpha=0.3, color='lightgreen')
ax.axhline(avg_price, color='red', linestyle='--', linewidth=2, label=f'Average: ${avg_price:.2f}')

# Highlight best and worst months
best_month = monthly_pred_df.loc[monthly_pred_df['predicted_price'].idxmax()]
worst_month = monthly_pred_df.loc[monthly_pred_df['predicted_price'].idxmin()]

ax.scatter([best_month['month']], [best_month['predicted_price']], 
          color='gold', s=300, zorder=5, edgecolor='black', linewidth=2)
ax.scatter([worst_month['month']], [worst_month['predicted_price']], 
          color='gray', s=300, zorder=5, edgecolor='black', linewidth=2)

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Predicted Price ($)', fontsize=12)
ax.set_title('Monthly Price Predictions for Your Property', fontsize=14, fontweight='bold')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBest Month: {best_month['month_name']} (${best_month['predicted_price']:.2f})")
print(f"Worst Month: {worst_month['month_name']} (${worst_month['predicted_price']:.2f})")
print(f"Price Difference: ${best_month['predicted_price'] - worst_month['predicted_price']:.2f}")

## 6. Generate Actionable Insights

In [ ]:
# Generate comprehensive insights
insights = generate_actionable_insights(df, predictor, YOUR_PROPERTY)

print("=" * 60)
print("ACTIONABLE INSIGHTS AND RECOMMENDATIONS")
print("=" * 60)

In [ ]:
# Market Comparison
print("\n1. MARKET COMPARISON")
print("-" * 60)

market_comp = insights['market_comparison']
market_stats = market_comp['market_stats']

print(f"\nSimilar Properties Analyzed: {market_comp['similar_properties_count']}")
print(f"\nMarket Statistics:")
print(f"  Average Price: ${market_stats['average_price']:.2f}")
print(f"  Median Price: ${market_stats['median_price']:.2f}")
print(f"  Price Range: ${market_stats['min_price']:.2f} - ${market_stats['max_price']:.2f}")
print(f"  25th Percentile: ${market_stats['percentile_25']:.2f}")
print(f"  75th Percentile: ${market_stats['percentile_75']:.2f}")
print(f"\nAverage Market Rating: {market_comp['average_rating']:.2f}")
print(f"Average Reviews Count: {market_comp['average_reviews']:.0f}")
print(f"Average Occupancy Rate: {market_comp['average_occupancy']:.1f}%")

In [ ]:
# Revenue Projection
print("\n2. REVENUE PROJECTION")
print("-" * 60)

revenue_proj = insights['revenue_projection']

print(f"\nAnnual Projections:")
print(f"  Gross Revenue: ${revenue_proj['annual_gross_revenue']:,.2f}")
print(f"  Est. Expenses (25%): -${revenue_proj['estimated_expenses']:,.2f}")
print(f"  Net Revenue: ${revenue_proj['estimated_net_revenue']:,.2f}")
print(f"  Avg Monthly Revenue: ${revenue_proj['average_monthly_revenue']:,.2f}")

# Show monthly breakdown
print(f"\nMonthly Revenue Breakdown:")
revenue_df = pd.DataFrame(revenue_proj['monthly_revenues'])
print(revenue_df[['month_name', 'nightly_rate', 'occupancy_rate', 'days_booked', 'revenue']].to_string(index=False))

In [ ]:
# Visualize revenue projection
revenue_df = pd.DataFrame(revenue_proj['monthly_revenues'])

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Monthly Revenue Projection', 'Occupancy Rate Trend')
)

# Monthly revenue bar chart
fig.add_trace(
    go.Bar(x=revenue_df['month_name'], y=revenue_df['revenue'], 
           name='Revenue', marker_color='steelblue',
           text=revenue_df['revenue'].apply(lambda x: f'${x:,.0f}'),
           textposition='auto'),
    row=1, col=1
)

# Occupancy rate line chart
fig.add_trace(
    go.Scatter(x=revenue_df['month_name'], y=revenue_df['occupancy_rate'],
               mode='lines+markers', name='Occupancy Rate',
               marker=dict(size=10, color='red'),
               line=dict(width=3, color='red')),
    row=1, col=2
)

fig.update_xaxes(title_text="Month", row=1, col=1)
fig.update_xaxes(title_text="Month", row=1, col=2)
fig.update_yaxes(title_text="Revenue ($)", row=1, col=1)
fig.update_yaxes(title_text="Occupancy Rate (%)", row=1, col=2)

fig.update_layout(height=500, showlegend=False, template='plotly_white')
fig.show()

In [ ]:
# Optimal Features Analysis
print("\n3. OPTIMAL FEATURES ANALYSIS")
print("-" * 60)

optimal = insights['optimal_features']

print(f"\nProperty Type Performance:")
for prop in optimal['property_types']:
    print(f"  {prop['type']:20s}: ${prop['average_price']:.2f} ({prop['count']} properties)")

print(f"\nBedroom Price Analysis:")
for bed in optimal['bedrooms']:
    print(f"  {bed['bedrooms']} Bedroom(s): ${bed['average_price']:.2f}")

print(f"\nPremiums:")
print(f"  Superhost Premium: +${optimal['superhost_premium']:.2f} per night")
print(f"  High Rating Premium: +${optimal['rating_premium']:.2f} per night")

In [ ]:
# Recommendations
print("\n4. ACTIONABLE RECOMMENDATIONS")
print("=" * 60)

for i, rec in enumerate(insights['recommendations'], 1):
    print(f"\n[{i}] {rec['title']}")
    print(f"    Category: {rec['category']}")
    print(f"    Priority: {rec['priority']}")
    print(f"    Description: {rec['description']}")
    print(f"    Action: {rec['action']}")

## 7. What-If Scenarios

In [ ]:
# Test different scenarios
print("=" * 60)
print("WHAT-IF SCENARIO ANALYSIS")
print("=" * 60)

scenarios = [
    {'name': 'Current Setup', 'bedrooms': 2, 'is_superhost': 1, 'rating': 4.5},
    {'name': '3 Bedrooms', 'bedrooms': 3, 'is_superhost': 1, 'rating': 4.5},
    {'name': 'Without Superhost', 'bedrooms': 2, 'is_superhost': 0, 'rating': 4.5},
    {'name': 'Lower Rating (4.0)', 'bedrooms': 2, 'is_superhost': 1, 'rating': 4.0},
    {'name': 'Premium (3 bed + 5.0 rating)', 'bedrooms': 3, 'is_superhost': 1, 'rating': 5.0}
]

scenario_results = []

for scenario in scenarios:
    features = YOUR_PROPERTY.copy()
    features.update(scenario)
    features['month'] = 6  # June average
    
    predicted_price = predictor.predict(pd.DataFrame([features]))[0]
    annual_revenue = predicted_price * 365 * 0.70  # 70% occupancy
    
    scenario_results.append({
        'Scenario': scenario['name'],
        'Nightly_Price': predicted_price,
        'Annual_Revenue': annual_revenue
    })
    
    print(f"\n{scenario['name']:30s}")
    print(f"  Predicted Price: ${predicted_price:.2f}/night")
    print(f"  Est. Annual Revenue: ${annual_revenue:,.2f}")

scenario_df = pd.DataFrame(scenario_results)

In [ ]:
# Visualize scenarios
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Nightly price comparison
axes[0].barh(scenario_df['Scenario'], scenario_df['Nightly_Price'], 
            color='steelblue', edgecolor='black')
axes[0].set_xlabel('Nightly Price ($)', fontsize=12)
axes[0].set_title('Scenario Price Comparison', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3, axis='x')

# Annual revenue comparison
axes[1].barh(scenario_df['Scenario'], scenario_df['Annual_Revenue']/1000, 
            color='coral', edgecolor='black')
axes[1].set_xlabel('Annual Revenue ($1000s)', fontsize=12)
axes[1].set_title('Scenario Revenue Comparison', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 8. Summary and Export

In [ ]:
# Create summary report
print("=" * 60)
print("EXECUTIVE SUMMARY")
print("=" * 60)

print(f"\nLocation: ({LOCATION_CONFIG['latitude']}, {LOCATION_CONFIG['longitude']})")
print(f"Analysis Radius: {LOCATION_CONFIG['radius_km']} km")
print(f"Properties Analyzed: {len(df)}")

print(f"\nYour Property Configuration:")
print(f"  Type: {YOUR_PROPERTY['property_type']}")
print(f"  Bedrooms: {YOUR_PROPERTY['bedrooms']}")
print(f"  Bathrooms: {YOUR_PROPERTY['bathrooms']}")
print(f"  Rating: {YOUR_PROPERTY['rating']}")

print(f"\nMarket Statistics:")
print(f"  Market Average Price: ${market_stats['average_price']:.2f}/night")
print(f"  Your Predicted Average: ${avg_price:.2f}/night")
print(f"  Position: {((avg_price / market_stats['average_price'] - 1) * 100):+.1f}% vs market")

print(f"\nRevenue Projection:")
print(f"  Annual Gross Revenue: ${revenue_proj['annual_gross_revenue']:,.2f}")
print(f"  Net Revenue (after 25% expenses): ${revenue_proj['estimated_net_revenue']:,.2f}")
print(f"  Monthly Average: ${revenue_proj['average_monthly_revenue']:,.2f}")

print(f"\nBest Performing Month: {best_month['month_name']} (${best_month['predicted_price']:.2f})")
print(f"Slowest Month: {worst_month['month_name']} (${worst_month['predicted_price']:.2f})")

print(f"\nModel Performance:")
print(f"  Best Model: {metrics['best_model']}")
print(f"  R² Score: {metrics['metrics'][metrics['best_model']]['r2_score']:.4f}")
print(f"  RMSE: ${metrics['metrics'][metrics['best_model']]['rmse']:.2f}")

In [ ]:
# Export results to CSV
monthly_pred_df.to_csv('monthly_price_predictions.csv', index=False)
revenue_df.to_csv('revenue_projections.csv', index=False)
scenario_df.to_csv('scenario_analysis.csv', index=False)

print("\nResults exported to:")
print("  - monthly_price_predictions.csv")
print("  - revenue_projections.csv")
print("  - scenario_analysis.csv")

In [ ]:
# Save the trained model
predictor.save_model('airbnb_price_model.pkl')
print("\nModel saved to: airbnb_price_model.pkl")